<h1>Boeing Final Assembly Rollout & Parking Optimization</h1>

<h2>User-Defined Values</h2>
<h3>1. Initialize for local machine</h3>

In [50]:
# user def / filepaths

user = 'jag'

userpaths = {
    'jag': '/Users/jishnughosh/Documents/GitHub/Boeing-Final-Assembly-and-Rollout-Optimization',
    'ksg': '',
    'mpg': '/Users/mpgen/OneDrive - purdue.edu/PURDUE/Spring 26/IE 431 - Personal',
    'da': '',
    'lah': ''
}

rootpath = userpaths[user] # overwrite if necessary

filepaths = {
    'FAstatus': '/input/FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': '/input/FGI_Locations_wPriority.xlsx',
    'Nodes': '/input/Nodes.xlsx'
}

<h3>2. Assumed Values</h3>
<p><strong>Changeable parameters:</strong></p>
<ul>
    <li><code>FGI_STAFFING_BYSHIFT</code>: Number of employees on each team for each shift</li>
    <li><code>FGI_CPJ</code>: Cost per job for each FGI team (# manhours required to complete 1 BTG task</li>
    <li><code>CENTERLINES</code>: Planes that must be moved for a move to occur starting at each location</li>

In [51]:
# algorithm toggles

FGI_STAFFING_BYSHIFT = { # FGI team: # manhours / 8hr shift
            1: {'structure': 16, 'systems': 60, 'declam': 16, 'test': 18},
            2: {'structure': 6, 'systems': 14, 'declam': 4, 'test': 0},
            3: {'structure': 0, 'systems': 0, 'declam': 0, 'test': 0}
        }

FGI_CPJ = { # FGI team: # manhours consumed / 1 BTG completed
        'structure': 6,
        'systems': 3.5,
        'declam': 6,
        'test': 8
    }

CENTERLINES = {
    'A1': None,
    'A2': None,
    'A3': None,
    'A4': None,
    'A5': None,
    'A6': None,
    'A7': None,
    'A8': None,
    'A9': ['C1'],
    'A10': ['C1', 'C2'],
    'BSC1': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'BSC2': ['C1', 'C2', 'C3', 'C3.5', 'C4', 'C5'],
    'C1': None,
    'C2': ['C1'],
    'C3': ['C1', 'C2'],
    'C3.5': ['C1', 'C2', 'C3'],
    'C4': ['C1', 'C2', 'C3', 'C3.5'],
    'C5': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'CR1': ['CR3', 'CR2'],
    'CR2': ['CR3'],
    'CR3': None,
    'D1': None,
    'D2': None,
    'F1': ['C1', 'C2'],
    'F2': ['C1', 'C2'],
    'L4': None,
    'L5': ['L4'],
    'Spur': None,
    'T1': None,
    'T2': None,
    'T3': None,
    'T4': None
}

FGI_TEAMS = ['structure', 'systems', 'declam', 'test']
BTG_TYPES = ['tot', 'p0', 'p1', 'p2', 'p3', 'engines', 'doors', 'test']

BTG_TYPES_BY_LABOR = { # relationships ONLY, NOT 1-to-1 btg conversion
    'structure': ['tot'],
    'systems': ['p2'],
    'declam': ['p3', 'engines'],
    'test': ['test']
}
FGI_TEAMS_BY_BTG_TYPE = { # relationships ONLY, NOT 1-to-1 btg conversion
    'tot': ['structure', 'systems', 'declam', 'test'],
    'FGI_tot': ['structure'],
    'p2': ['systems'],
    'p3': ['declam'],
    'engines': ['declam'],
    'test': ['test']
}
for type in BTG_TYPES:
    if type not in FGI_TEAMS_BY_BTG_TYPE.keys():
        FGI_TEAMS_BY_BTG_TYPE[type] = None


<p>|\==><i>NOTE: THESE PARAMETERS WILL BE OVERWRITTEN IF INPUT_TYPE IS SET TO FGI_LIVESTATE</p>

<h2>Initialization</h2>
<h3>1. Import Libraries</h3>
<p><strong>Required Installations:</strong></p>
<ul>
    <li>pandas</li>
    <li>numpy</li>
    <li>datetime</li>
    <li>matplotlib.pyplot</li>
    <li>seaborn</li>
    <li>math</li>
</ul>

In [52]:
import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import math
from math import dist
# import ast
# import heapq
# import re
# import os

<h3>2. Run Conditions</h3>
<p>These parameters set the conditions under which the algorithm will generate the schedule. Be sure that this is updated on each run of the main placement algorithm</p>

In [53]:
# a) planning horizon
STARTDATE='2024-06-10'
ENDDATE='2026-03-10'
FORECAST_UNTIL_COMPLETION = True

# b) location parameters

NEW_FA_ONLINE = False
DISCONTINUED_LOCATIONS = ['S1', 'S2', 'S3', 'Spur']
INCLUDE_FA_LOCATIONS = False

# c) schedule parameters
PLANNED_BUFFER_DAYS = []
IMPORT_PAINT_SCHEDULE = False


# d) input parameters
input_types = [
    'FARO_SCORECARD_WITH_MILESTONES',
    'FGI_LIVESTATE'
]
INPUT_TYPE = input_types[0]

# e) output/export parameters
EXPORT_TO_EXCEL = True
EXPORT_PATH = '/output/'
CODECELL_OUTPUT = True
EXPORT_TO_FGI_LIVESTATE = True


<p><i>|==> Initialize based on run conditions </i></p>

In [54]:
# dataframe builder functions
def merge_APdata(faro_scorecard, p3_milestones, tankClosure):
    ## organize milestone/tankClosure by LN
    tank_lookup = (
        tankClosure[['LINE_NUMBER', 'TankStatus']]
        .drop_duplicates(subset='LINE_NUMBER')
        .rename(columns={'LINE_NUMBER': 'LN'})
    )
    p3_lookup = (
        p3_milestones
        .rename(columns={'P': 'LN'})
        .copy()
    )

    faro_scorecard = faro_scorecard.copy()
    faro_scorecard['LN'] = pd.to_numeric(faro_scorecard['LN'], errors='coerce')

    tank_lookup['LN'] = pd.to_numeric(tank_lookup['LN'], errors='coerce')
    p3_lookup['LN'] = pd.to_numeric(p3_lookup['LN'], errors='coerce')

    ## store LN as str
    faro_scorecard['LN'] = faro_scorecard['LN'].astype(str)
    tank_lookup['LN'] = tank_lookup['LN'].astype(str)
    p3_lookup['LN'] = p3_lookup['LN'].astype(str)

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    return ap_df

def build_ap_df(faro_scorecard, p3_milestones, tankClosure):
    ap_data = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

    rows = []

    for _, row in ap_data.iterrows():
        ln = int(row['LN'])

        rows.append({
            'LN': ln,
            'FA RO': row['FA RO'],
            'FA RO to B1R': row['FA RO to B1R'],
            'Total Counters': row['Total Counters'],
            'TankStatus': row['TankStatus'],
            'Ceilings': row['Ceilings'],
            'Initial Tests Run': row['Initial Tests Run'],

            # BTG attributes
            'BTG_tot': row['Total BTG'],
            'BTG_p0': row['P0 BTG'],
            'BTG_p1': row['P1 BTG'],
            'BTG_p2': row['P2 BTG'],
            'BTG_p3': row['P3 BTG'],
            'BTG_engines': row['Engines BTG'],
            'BTG_doors': row['Doors BTG'],
            'BTG_test': row['Test BTG'],

            # P3 milestone attributes
            'P3_Engine Hang': row['Engine Hang'],
            'P3_Flight Controls': row['Flight Controls'],
            'P3_Gear Swing': row['Gear Swing'],
            'P3_Medium Pressure Test': row['Medium Pressure Test'],
            'P3_Oil On': row['Oil On'],
            'P3_Power On': row['Power On'],
            'P3_Engine_Type': row['Engine_Type'],
            'P3_Milestone_Completion_Percentage': row['Milestone_Completion_Percentage'],

            # Shake attributes
            'shakes_complete': row['shakes_completed'],
            'shakes_req': row['shakes_req']
        })

    return pd.DataFrame(rows)

def build_location_df(fa_priority, centerlines):
    rows = []
    seen_locations = set()

    for _, row in fa_priority.iterrows():
        loc = row['Location']
        seen_locations.add(loc)
        centerline_deps = centerlines.get(loc, None)
        

        rows.append({
            'Location': loc,
            'Future State Priority': row['Future State Priority'],
            'Date Online': row['Date Online'],
            'Owner': row['Owner'],
            'tooling_jacking': row['Jacking'] == 'Y',
            'tooling_wings': row['Wings'] == 'Y',
            'tooling_tankClosure': row['Tank Closure'] == 'Y',
            'centerline_dependencies': None if centerline_deps is None else ', '.join(centerline_deps),
            # 'obstructions': None,
            # 'notes': None
        })

    return pd.DataFrame(rows)

def build_labor_df(fgi_staffing_byshift, fgi_cpj, startdate, enddate):
    rows = []

    # FGI staffing by shift
    for shift, teams in fgi_staffing_byshift.items():
        for team, manhours in teams.items():
            rows.append({
                'category': 'FGI_STAFFING_BYSHIFT',
                'shift': shift,
                'team': team,
                'value': manhours
            })

    # FGI CPJ
    for team, cpj in fgi_cpj.items():
        rows.append({
            'category': 'FGI_CPJ',
            'shift': None,
            'team': team,
            'value': cpj
        })

    # Dates
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'STARTDATE',
        'value': startdate
    })
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'ENDDATE',
        'value': enddate
    })

    return pd.DataFrame(rows)


# build dataframes from FARO scorecare with milestones
if INPUT_TYPE == 'FARO_SCORECARD_WITH_MILESTONES':
    def clean_FAstatus(faro_scorecard, tankClosure, p3_milestones):
        ## FARO Scorecard Cleaning
        # only take rows with valid LNs
        faro_scorecard = faro_scorecard[pd.to_numeric(faro_scorecard['LN'], errors='coerce').notna()].copy()
        # remove unnamed columns
        faro_scorecard = faro_scorecard.loc[:, ~faro_scorecard.columns.astype(str).str.contains(r'^Unnamed')]

        # split zone shakes column into two columns: completed/required
        faro_scorecard[['shakes_completed', 'shakes_req']] = faro_scorecard['Zone Shakes'].astype(str).str.split('/', expand=True)
        # only take 
        faro_scorecard['p3_milestones'] = faro_scorecard['P3 Milestones'].astype(str).str.split('/').str[0]

        # set datatypes
        faro_scorecard = (
            faro_scorecard.assign(
                LN=lambda df: pd.to_numeric(df['LN'], errors='coerce').astype(int),
                **{
                    'FA RO to B1R': lambda df: pd.to_numeric(df['FA RO to B1R'], errors='coerce'),
                    'Total Counters': lambda df: pd.to_numeric(df['Total Counters'], errors='coerce').fillna(0),
                    'Total BTG': lambda df: pd.to_numeric(df['Total BTG'], errors='coerce').fillna(0),
                    'P0 BTG': lambda df: pd.to_numeric(df['P0 BTG'], errors='coerce').fillna(0),
                    'P1 BTG': lambda df: pd.to_numeric(df['P1 BTG'], errors='coerce').fillna(0),
                    'P2 BTG': lambda df: pd.to_numeric(df['P2 BTG'], errors='coerce').fillna(0),
                    'P3 BTG': lambda df: pd.to_numeric(df['P3 BTG'], errors='coerce').fillna(0),
                    'Engines BTG': lambda df: pd.to_numeric(df['Engines BTG'], errors='coerce').fillna(0),
                    'Doors BTG': lambda df: pd.to_numeric(df['Doors BTG'], errors='coerce').fillna(0),
                    'Test BTG': lambda df: pd.to_numeric(df['Test BTG'], errors='coerce').fillna(0),
                    'Tank Closure': lambda df: df['Tank Closure'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int),
                    'Ceilings': lambda df: pd.to_numeric(df['Ceilings'], errors='coerce').fillna(0),
                    'Initial Tests Run': lambda df: (
                        df['Initial Tests Run'].astype(str)
                        .str.replace('%', '', regex=False)
                        .replace('nan', 0)
                        .replace('', 0)
                        .astype(float) / 100
                    ),
                    'shakes_completed': lambda df: pd.to_numeric(df['shakes_completed'], errors='coerce').fillna(0).astype(int),
                    'shake_req': lambda df: pd.to_numeric(df['shakes_req'], errors='coerce').fillna(0).astype(int),
                    'p3_milestones': lambda df: pd.to_numeric(df['p3_milestones'], errors='coerce').fillna(0).astype(int),
                }
            )
        )

        ## TANK CLOSURE Cleaning
        tankClosure['LINE_NUMBER'] = pd.to_numeric(tankClosure['LINE_NUMBER'], errors='coerce')
        tankClosure['Complete_Jobs'] = pd.to_numeric(tankClosure['Complete_Jobs'], errors='coerce').fillna(0)
        tankClosure['Total_Jobs'] = pd.to_numeric(tankClosure['Total_Jobs'], errors='coerce').fillna(0)
        tankClosure['TankStatus'] = tankClosure['TankStatus'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int)

        ## P3 Milestone Cleaning
        p3_milestones['Engine_Type'] = p3_milestones['Milestone'].astype(str).str.extract(r'\((.*?)\)')
        p3_milestones['Milestone'] = p3_milestones['Milestone'].astype(str).str.replace(r'\s*\(.*?\)', '', regex=True)
        p3_milestones = (
            p3_milestones.pivot_table(index='P', columns='Milestone', values='Completed_Jobs', aggfunc='sum')
            .assign(
                Engine_Type=p3_milestones.groupby('P')['Engine_Type'].first(),
                Milestone_Completion_Percentage=p3_milestones.groupby('P')['STATUS (1 Complete, 0 Still Open)'].mean()
            )
            .fillna(-1)
            .reset_index()
        )

        return faro_scorecard, tankClosure, p3_milestones

    def clean_nodeData(nodes, node_adj):
        ## clean nodes
        nodes = (
            nodes
            .drop(columns=[c for c in nodes.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                node_id=lambda df: df['node_id'].astype('string').str.strip(),
                x=lambda df: pd.to_numeric(df['x'], errors='coerce'),
                y=lambda df: pd.to_numeric(df['y'], errors='coerce'),
                type=lambda df: df['type'].astype('string').str.strip(),
                req_centerline=lambda df: df['req_centerline'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['node_id', 'x', 'y'])
            .reset_index(drop=True)
        )

        ## clean node_adj 
        node_adj = (
            node_adj
            .drop(columns=[c for c in node_adj.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                from_node=lambda df: df['from_node'].astype('string').str.strip(),
                to_node=lambda df: df['to_node'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['from_node', 'to_node'])
            .drop_duplicates()
            .reset_index(drop=True)
        )

        return nodes, node_adj

    def import_data(rootpath, filepaths = {'FAstatus': 'FA_Status_FGI_Handoff.xlsx','FGI_Locations': 'FGI_Locations_wPriority.xlsx','Nodes': 'Nodes.xlsx'}):
        
        ## FA STATUS DATA IMPORT
        path = rootpath + filepaths['FAstatus']
        # read FARO scorecard as df
        faro_scorecard = pd.read_excel(path, sheet_name='FARO_Scorecard',header=2,engine='openpyxl')
        tankClosure = pd.read_excel(path,sheet_name='Tank_Closure_Detail',engine='openpyxl')
        p3_milestones = pd.read_excel(path, sheet_name='P3 Milestone Detail',engine='openpyxl')

        ## FGI LOCATION DATA IMPORT
        path = rootpath + filepaths['FGI_Locations']
        fa_priority = pd.read_excel(path, sheet_name='FA Priority', header=1,engine='openpyxl')

        ## NODES DATA IMPORT
        path = rootpath + filepaths['Nodes']
        nodes = pd.read_excel(path,sheet_name='Node',engine='openpyxl')
        node_adj = pd.read_excel(path, sheet_name='adjacency',engine='openpyxl')

        ## DATA CLEANING
        faro_scorecard, tankClosure, p3_milestones = clean_FAstatus(faro_scorecard, tankClosure, p3_milestones)

        fa_priority = (
            fa_priority
            .drop(columns=[c for c in fa_priority.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .reset_index(drop=True)
        )

        nodes, node_adj = clean_nodeData(nodes, node_adj)

        return faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj

    faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
    ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

    location_df = build_location_df(
        fa_priority=fa_priority,
        centerlines=CENTERLINES
    )

    labor_df = build_labor_df(
        fgi_staffing_byshift=FGI_STAFFING_BYSHIFT,
        fgi_cpj=FGI_CPJ,
        startdate=STARTDATE,
        enddate=ENDDATE
    )


    staffing_df = labor_df[labor_df['category'] == 'FGI_STAFFING_BYSHIFT'].copy()
    staffing_df['shift'] = staffing_df['shift'].astype(int)
    staffing_df['value'] = pd.to_numeric(staffing_df['value'])
    FGI_STAFFING_BYSHIFT = {
        shift: skill.set_index('team')['value'].to_dict()
        for shift, skill in staffing_df.groupby('shift', sort=True)
    }

    FGI_CPJ = (
        labor_df[labor_df['category'] == 'FGI_CPJ']
        .assign(value=lambda df: pd.to_numeric(df['value']))
        .set_index('team')['value']
        .to_dict()
    )



# build dataframes from FGI_LIVESTATE
if INPUT_TYPE == 'FGI_LIVESTATE':

    def load_live_state(path='FGI_liveState.xlsx'):
        required_columns = {
            'ap_data': [
                'LN', 'FA RO', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
                'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
                'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
                'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
                'P3_Oil On', 'P3_Power On', 'P3_Engine_Type',
                'P3_Milestone_Completion_Percentage', 'shakes_complete', 'shakes_req'
            ],
            'location_data': [
                'Location', 'Future State Priority', 'Date Online', 'Owner',
                'tooling_jacking', 'tooling_wings', 'tooling_tankClosure',
                'centerline_dependencies', 'obstructions', 'notes'
            ],
            'labor_data': ['category', 'shift', 'team', 'value']
        }

        numeric_ap_columns = [
            'LN', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
            'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
            'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
            'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
            'P3_Oil On', 'P3_Power On', 'P3_Milestone_Completion_Percentage',
            'shakes_complete', 'shakes_req'
        ]

        tooling_columns = ['tooling_jacking', 'tooling_wings', 'tooling_tankClosure']

        def _normalize_bool(value):
            if pd.isna(value):
                return False
            if isinstance(value, (bool, np.bool_)):
                return bool(value)
            if isinstance(value, (int, float)):
                return value != 0

            value_str = str(value).strip().lower()
            if value_str in {'true', 't', 'yes', 'y', '1'}:
                return True
            if value_str in {'false', 'f', 'no', 'n', '0', ''}:
                return False

            raise ValueError(f'Invalid boolean value in location_data: {value}')

        if not os.path.exists(path):
            raise ValueError(f'Live state workbook not found: {path}')

        try:
            workbook = pd.ExcelFile(path, engine='openpyxl')
        except FileNotFoundError as exc:
            raise ValueError(f'Live state workbook not found: {path}') from exc
        except Exception as exc:
            raise ValueError(f'Unable to open live state workbook: {path}') from exc

        missing_sheets = [sheet for sheet in required_columns if sheet not in workbook.sheet_names]
        if missing_sheets:
            raise ValueError(
                f"Missing required sheet(s) in {path}: {', '.join(missing_sheets)}"
            )

        ap_df = pd.read_excel(workbook, sheet_name='ap_data')
        location_df = pd.read_excel(workbook, sheet_name='location_data')
        labor_df = pd.read_excel(workbook, sheet_name='labor_data')

        frames = {
            'ap_data': ap_df,
            'location_data': location_df,
            'labor_data': labor_df
        }

        for sheet_name, expected_columns in required_columns.items():
            missing_columns = [col for col in expected_columns if col not in frames[sheet_name].columns]
            if missing_columns:
                raise ValueError(
                    f"Missing required column(s) in sheet '{sheet_name}': {', '.join(missing_columns)}"
                )

        ap_df = ap_df.copy()
        ap_df['FA RO'] = pd.to_datetime(ap_df['FA RO'], errors='coerce')
        for column in numeric_ap_columns:
            ap_df[column] = pd.to_numeric(ap_df[column], errors='coerce')

        if ap_df['LN'].notna().all():
            ap_df['LN'] = ap_df['LN'].astype('Int64')

        location_df = location_df.copy()
        location_df['Location'] = location_df['Location'].astype('string').str.strip()
        location_df = location_df[location_df['Location'].notna() & location_df['Location'].ne('')].reset_index(drop=True)
        for column in tooling_columns:
            location_df[column] = location_df[column].map(_normalize_bool).astype(bool)

        labor_df = labor_df.copy()
        labor_df['shift'] = pd.to_numeric(labor_df['shift'], errors='coerce').astype('Int64')

        return ap_df, location_df, labor_df
    
    ap_df, location_df, labor_df = load_live_state(path='FGI_liveState.xlsx')
    

if EXPORT_TO_FGI_LIVESTATE:

    output_path = rootpath + EXPORT_PATH + 'FGI_liveState.xlsx'

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        ap_df.to_excel(writer, sheet_name='ap_data', index=False)
        location_df.to_excel(writer, sheet_name='location_data', index=False)
        labor_df.to_excel(writer, sheet_name='labor_data', index=False)

    if CODECELL_OUTPUT: print(output_path)

if CODECELL_OUTPUT:
    def line(char='—', length = 100):
        line = char
        while len(line) < length:
            line += char
        print(line+'\n')
    # HEADER_NUMBER = 0
    def header(text, char='_', length=100):
        # HEADER_NUMBER = HEADER_NUMBER + 1
        padding = [f'{char}| ', f' |{char}']
        line = f'{char}| {text} |{char}'
        while len(line) < length:
            padding = [char+padding[0], padding[1]+char]
            line = padding[0] + text + padding[1]

        print(line)
        


/Users/jishnughosh/Documents/GitHub/Boeing-Final-Assembly-and-Rollout-Optimization/output/FGI_liveState.xlsx


<p>|\<i>==> Now, dataframes have been built to describe APs, Locations, and Labor Allocation pararameters to build the schedule</i></p>

<h2>Scheduler</h2>
<h3>1. Class Initialization</h3>
<h4>a) Definition</h4>

In [55]:
class AP:
    ## ————————————OBJECT INITIALIZATION——————————————————————————————————————————————————————————————————————————
    ## Initialize AP with FARO attributes
    def __init__(self,LN,faro,toB1R,counters,btg=None,tankClosed=False,ceilings=0,testsRun=0,p3_milestones=None,shakes=None):
        # FA attributes at RO
        self.LN=LN
        self.faro=faro
        self.toB1R=toB1R
        self.counters=counters
        self.btg=btg if btg is not None else {"tot":0,"p0":0,"p1":0,"p2":0,"p3":0,"engines":0,"doors":0,"test":0}
        self.tankClosed=tankClosed
        self.ceilings=ceilings
        self.testsRun=testsRun
        self.p3milestones=p3_milestones if p3_milestones is not None else {'tot':6,'complete':0}
        self.shakes = shakes if shakes is not None else {'complete': 0, 'req': 4}
        
        # scheduling attributes
        self.schedule={}
        self.moveReq=False
        self.path=[]

        # state attributes
        self.Location = None
        self.taskState = 'FA'


        ##_ _ _ _ _ _ _ _ _ _ _ _| convert FARO BTG => FGI BTG |_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
        self.FGI_btg = {'FGI_tot': self.btg['tot'] - self.btg['p0'] - self.btg['p1'], 
                        'structure': math.ceil(0.1 * (self.btg['tot'] - self.btg['p0'] - self.btg['p1'])), 
                        'systems': math.ceil(self.btg['p2']),
                        'declam': math.ceil(self.btg['p3'] + self.btg['engines']),
                        'test': self.btg['test']
                        }
        self.taskCompletion = {
            'move': self.moveReq,
            'compass': False,
            'paint': False,
            'structure': False if self.FGI_btg['structure'] > 0 else True,
            'systems': False if self.FGI_btg['systems'] > 0 else True,
            'declam': False if self.FGI_btg['declam'] > 0 else True,
            'test': False if self.FGI_btg['test'] > 0 else True
        }

    ## ————————————| GET METHODS |——————————————————————————————————————————————————————————————————————————
    # clean attributes into normalized datatypes for use within algorithm
    def get_LN(self): return str(self.LN) # —
    def get_FAROdate(self): return pd.to_datetime(self.faro)
    def get_days_toB1R(self): return pd.Timedelta(days=self.toB1R)
    def get_counters(self): return int(self.counters)
    def get_BTG(self, category): return pd.to_numeric(self.btg[category]) if category in self.btg.keys() else False
    #def get_tankClosure(self): return self.tankClosed # —   —   —   —   —   UNUSED, always boolean
    def get_percent_ceilings(self): return pd.to_numeric(self.ceilings)
    def get_percent_testsRun(self): return pd.to_numeric(self.testsRun)
    #def get_p3_milestones(self): return self.p3milestones # —   —   —   —   —   UNUSED, dict
    def get_daystoB1R(self): return pd.Timedelta(days=self.toB1R)
    def get_fgi_btg(self, team): return pd.to_numeric(self.FGI_btg[team]) if team in self.FGI_btg.keys() else False
    
    ## ————————————| SCHEDULING METHODS |——————————————————————————————————————————————————————————————————————————
    
    def set_taskState(self, state):
        AP_TASK_STATES = ['FA', 'RO', 'paint', 'compass', 'shake', 'btg_completion', 'tankClosure', 'toDC', 'exit', 'delivered']
        if state in AP_TASK_STATES:
            self.taskState = state
        else:
            if CODECELL_OUTPUT:
                header('INVALID METHOD CALL')
                print(f'LN: {self.LN} taskState set to {state}')
                line()
            return False

    def isMoveReq(self): return self.moveReq

    def requireMove(self): 
        self.moveReq = True
        if CODECELL_OUTPUT:
            header('MOVE REQUIRED')
            current_loc = self.Location.name if self.Location is not None else None
            print(f'LN: {self.LN} requires move from {current_loc}\nTask State: {self.taskState}')
            line()

    def complete_BTG(self, category, btg_count=0, byLabor=False):
        ## input checking
        if btg_count is None or btg_count <= 0:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print('tried to complete btg on an AP when btg_count was 0 or no parameter given')
                print(f'LN: {self.LN} category: {category} btg_count: {btg_count}')
                line()
            return 0

        if not byLabor and category not in self.btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid BTG category when byLabor=False: {category}')
                print(f'valid categories: {list(self.btg.keys())}')
                line()
            return btg_count

        if byLabor and category not in self.FGI_btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid labor category when byLabor=True: {category}')
                print(f'valid categories: {list(self.FGI_btg.keys())}')
                line()
            return btg_count

        remaining_btg = btg_count

        ## complete BTG directly by BTG category
        if not byLabor:
            available = max(self.btg[category], 0)
            completed = min(available, remaining_btg)

            self.btg[category] -= completed
            self.btg['tot'] -= completed

            if category in FGI_TEAMS_BY_BTG_TYPE and FGI_TEAMS_BY_BTG_TYPE[category] is not None:
                for group in FGI_TEAMS_BY_BTG_TYPE[category]:
                    if group in self.FGI_btg:
                        self.FGI_btg[group] = max(self.FGI_btg[group] - completed, 0)

            remaining_btg -= done

        ## complete BTG by labor bucket
        else:
            available = max(self.FGI_btg[category], 0)
            completed = min(available, remaining_btg)

            self.FGI_btg[category] -= completed

            if category in BTG_TYPES_BY_LABOR:
                for btg_type in BTG_TYPES_BY_LABOR[category]:
                    if btg_type in self.btg:
                        self.btg[btg_type] = max(self.btg[btg_type] - completed, 0)

            self.btg['tot'] = max(self.btg['tot'] - completed, 0)
            remaining_btg -= completed

        return True, remaining_btg



    # def exit(self,date):
    #     self.Location = None
    #     self.schedule[date] = 'B1R'
    #     if CODECELL_OUTPUT:
    #         print('=================AP EXIT==================')
    #         print(f'AP {self.LN} exited to B1R on {date.strftime("%Y-%m-%d")}')
    #         print(f'Path taken: {self.path + ["B1R"]}')
    #         print(f'FA RO on {self.faro.strftime("%Y-%m-%d")}, days to B1R: {self.toB1R}')
    #         print(f'Expected B1R date:{pd.to_timedelta(self.toB1R, unit="D")+ pd.to_datetime(self.faro).strftime("%Y-%m-%d")}')
    #         print('==========================================')


    # def is_past_B1R(self, date):
    #     if date > (self.get_FAROdate() + self.get_daystoB1R()):
    #         return True
    #     else:
    #         return False
    

class Location:
    def __init__(self,priority,dateOnline,name,owner=None,tooling=None,centerlines=None):
        self.priority=priority

        if dateOnline == 'Now':
            self.isOnline = True
        elif dateOnline == 'At R10' and NEW_FA_ONLINE == True:
            self.isOnline = False
        else:
            self.isOnline = False
        
        self.name=name
        self.owner=owner
        self.tooling=tooling if tooling is not None else {'jacking':False,'wings':False,'tankClosure':False}
        # self.obstructions=obstructions if obstructions is not None else []
        # self.notes=notes if notes is not None else []
        self.centerlines = None
        if centerlines is not None:
            self.centerlines = [centerline.strip() for centerline in str(centerlines).split(',') if centerlines.strip()]
            
        self.schedule={}
        self.time_to = {}
        self.AP = None

    # def get_priority(self): return self.priority
    # def get_dateOnline(self): return self.dateOnline
    # def get_name(self): return self.location
    # def getName(self): return self.location
    # def get_owner(self): return self.owner
    # def get_tooling(self): return self.tooling
    # def get_obstructions(self): return self.obstructions
    # def get_notes(self): return self.notes
    # def get_schedule(self): return self.schedule
    # def get_occupiedBy(self): return self.occupiedBy

    def canUse(self,tool): 
        if tool in self.tooling.keys():
            return self.tooling[tool]
        
    def isAvailable(self): return True if self.AP == None else False

    def canPlace(self, ap):
        if self.isOnline:
            return self.isAvailable
        else:
            return False
    
    def assign(self, AP, date=None):
        self.AP = AP 
        if date is not None:
            self.schedule[date] = AP.get_LN()
        if self.AP is not None:
            header('ERROR', '~')
            print("AP assigned to unavailable location")
            line('~')

    def unassign(self,date=None):
        if CODECELL_OUTPUT:
            header('MOVE', ' - ')
            print(f'{self.AP.get_LN()} removed from location {self.name}')
            line(' - ')

        self.AP = None
        


    def obstructs(self,obstruction): return obstruction in self.obstructions
    def isHigherPriorityThan(self,other): return self.priority>other.priority
    def clear_schedule(self): self.schedule={}
    def set_time_to(self, other, move_time):
        self.time_to[other] = move_time

# =====================MAIN SCHEDULING CLASS===========================
## FGI manages all APs currently owned by FGI and all active locations within FGI
## currently also manages queues for moves, fgi tasks, and labor

class FGI:
    def __init__(self, labor, CPJ):
        self.labor = labor
        self.CPJ = CPJ
        self.APs = {}
        self.Locations = {}
        self.locations = self.Locations
        self.queues = {
            'move': [],
            'FGI task': {
                'compass': [],
                'paint': []
            },
            'labor': {
                'structure': [],
                'systems': [],
                'declam': [],
                'test': []
            }
        }
        self.schedule = {}
        self.chickenTracks = {}
        self.apStateRows = []
        self.shift = 0

    def get_queue(self, queue):
        if queue == 'all':
            return self.queues
        else:
            return self.queues[queue]

    def get_labor(self, team, shift=0):
        if self.shift == 0 and CODECELL_OUTPUT:
            header('ERROR', '~')
            print('Shift must be initialized for FGI object')
            line()
            return None
            

        if team == 'all':
            return self.labor[shift]
        else:
            return self.labor[shift][team]
        
    def get_btg_completion(self, team, shift=0):
        if shift == 0:
            header('ERROR', '~')
            return False
        else:
            return pd.to_numeric(self.labor[shift][team] / self.CPJ[team])
        
    def add_ap(self, ap, date=None):
        self.APs[ap.get_LN()] = ap
        for task in self.queues['FGI task']:
            # if not ap.taskCompletion[task] and ap.get_LN() not in self.queues['FGI task'][task]:
            self.queues['FGI task'][task].append(ap.get_LN())
        for team in self.queues['labor']:
            # if not ap.taskCompletion[team] and ap.get_LN() not in self.queues['labor'][team]:
            self.queues['labor'][team].append(ap.get_LN())

        ## place immediately if on FARO date
        if date is not None and date == ap.get_FAROdate():
            ap.set_taskState = 'RO'

            if ap.Location is None:
                ap.requireMove()
                if ap.get_LN() not in self.queues['move']:
                    self.queues['move'].insert(0, ap.get_LN())
        else:
            pass

        if CODECELL_OUTPUT:
            line()
            print(f'LN {LN} added to FGI')
            print(f'Current APs in FGI: {list(self.APs.keys())}')
            line()        

    def add_Location(self, name, location_obj):
        self.Locations[name] = location_obj
        self.chickenTracks[name] = None
        if CODECELL_OUTPUT:
            line()
            print(f'Location {name} added to FGI')
            print(f'Current locations in FGI: {list(self.Locations.keys())}')
            line()

    def calibrateCompass(self, LN):
        self.APs[LN].taskState = 'CR3'
        self.APs[LN].requireMove()
        self.queues['move'].append(LN)
        compass_obstructions = ['CR1', 'CR2']
        for loc in compass_obstructions:
            if self.chickenTracks[loc] is not None:
                obstructing_ap = self.chickenTracks[loc]
                self.queues['move'].insert(obstructing_ap.get_LN())
                if CODECELL_OUTPUT:
                    print(f'AP {obstructing_ap.get_LN()} added to move queue due to compass obstruction at {loc}')
        if self.APs[LN].centerlines_required() != None:
            for loc in self.APs[LN].centerlines_required():
                self.queues['move'].insert(LN)

    def assign_labor(self, team, max_btg_completion=30):
    
        btg_completed = 0
        worked_lns = []

        if team not in self.queues['labor']:
            raise ValueError(f'Invalid labor team: {team}')

        queue = self.queues['labor'][team]
        i = 0

        while i < len(queue) and btg_completed < max_btg_completion:
            LN = queue[i]
            ap = self.APs[LN]

            remaining_capacity = max_btg_completion - btg_completed
            success, remaining_btg = ap.complete_BTG(team, btg_count=remaining_capacity, byLabor=True)

            if success:
                done = remaining_capacity - remaining_btg
                btg_completed += done

                if done > 0:
                    worked_lns.append(LN)

            ## if this queue's work is now complete, remove LN from queue
            if ap.get_fgi_btg(team) <= 0:
                queue.pop(i)

                ## once a labor bucket completes, require move
                ap.requireMove()
                if LN not in self.queues['move']:
                    self.queues['move'].append(LN)
            else:
                i += 1

        return worked_lns, btg_completed
    
    def move_ap(self, LN, date=None):
        AP = self.APs[LN]

        current_loc = AP.Location
        destination = None

        for loc_name, loc in self.Locations.items():
            if current_loc is not None and loc.name == current_loc.name:
                continue
            if loc.canPlace(AP):
                destination = loc
                break

        if destination is None:
            return False

        if current_loc is not None:
            current_loc.unassign(date=date)
            self.chickenTracks[current_loc.name] = None

        destination.assign(AP, date=date)
        AP.Location = destination
        AP.moveReq = False
        AP.path.append(destination.name)
        self.chickenTracks[destination.name] = AP

        return True

    def process_move_queue(self, date=None):
        i = 0

        while i < len(self.queues['move']):
            LN = self.queues['move'][i]
            AP = self.APs[LN]

            moved = self.move_ap(LN, date=date)

            if moved or AP.isMoveReq() is False:
                self.queues['move'].pop(i)
            else:
                i += 1
    
    def record_day(self, date):
        self.schedule[date] = {}

        for loc_name, loc in self.Locations.items():
            self.schedule[date][loc_name] = None if loc.AP is None else loc.AP.get_LN()

        for LN, AP in self.APs.items():
            self.apStateRows.append({
                'Date': date,
                'LN': LN,
                'Location': None if AP.Location is None else AP.Location.name,
                'FGI_tot': AP.get_fgi_btg('FGI_tot'),
                'structure': AP.get_fgi_btg('structure'),
                'systems': AP.get_fgi_btg('systems'),
                'declam': AP.get_fgi_btg('declam'),
                'test': AP.get_fgi_btg('test'),
                'moveReq': AP.isMoveReq()
            })


<h4>b) <i>Functions</i>: Initialize AP

In [56]:

def init_APs(ap_df):
    return {
        str(row['LN']): AP(
            LN=int(row['LN']),
            faro=row['FA RO'],
            toB1R=row['FA RO to B1R'],
            counters=row['Total Counters'],
            btg={
                'tot': row['BTG_tot'],
                'p0': row['BTG_p0'],
                'p1': row['BTG_p1'],
                'p2': row['BTG_p2'],
                'p3': row['BTG_p3'],
                'engines': row['BTG_engines'],
                'doors': row['BTG_doors'],
                'test': row['BTG_test']
            },
            tankClosed=row['TankStatus'],
            ceilings=row['Ceilings'],
            testsRun=row['Initial Tests Run'],
            p3_milestones={
                'Engine Hang': row['P3_Engine Hang'],
                'Flight Controls': row['P3_Flight Controls'],
                'Gear Swing': row['P3_Gear Swing'],
                'Medium Pressure Test': row['P3_Medium Pressure Test'],
                'Oil On': row['P3_Oil On'],
                'Power On': row['P3_Power On'],
                'Engine_Type': row['P3_Engine_Type'],
                'Milestone_Completion_Percentage': row['P3_Milestone_Completion_Percentage']
            },
            shakes={'complete': row['shakes_complete'], 'req': row['shakes_req']}
        )
        for _, row in ap_df.iterrows()
    }

def init_Locations(location_df):
    Locations = {}
    for _, row in location_df.iterrows():
        if not pd.isna(row['Location']):
            Locations[row['Location']] = Location(
                priority=row['Future State Priority'],
                dateOnline=row['Date Online'],
                name=row['Location'],
                owner=row['Owner'],
                tooling={
                    'jacking': row['tooling_jacking'],
                    'wings': row['tooling_wings'],
                    'tankClosure': row['tooling_tankClosure']
                },
                centerlines = row['centerline_dependencies']
            )
    return Locations

<h3>2. Function Definitions</h3>

<h4>a) Job Completion & Duration Helpers</h4>

In [57]:
# def get_fgi_btg(btg):
#     fgi_tot = btg['tot'] - btg['p0'] - btg['p1']
#     return {
#         'fgi_tot': math.ceil(fgi_tot),
#         'structure': math.ceil(0.1 * fgi_tot),
#         'systems': math.ceil(btg['p2']),
#         'declam': math.ceil(btg['p3'] + btg['engines']),
#         'test': math.ceil(btg['test'])
#     }

# def calc_btg_completion(fgi, shift, team, manhours):
    
#     manhours = {
#         shift: {team: staffing * hours_per_shift for team, staffing in teams.items()}
#         for shift, teams in fgi_staffing.items()
#     }
#     btg_completion = {
#         shift: {team: manhours[shift][team] / cpj[team] for team in teams}
#         for shift, teams in manhours.items()
#     }
#     return manhours, btg_completion


<h4>b) Scheduling Helpers</h4>

In [58]:
def calibrate_compass(FGI, ln):
    pass

def isWorkday(date):
    return date.weekday() < 5  # Monday=0, Sunday=6



<h3>Main Loop — Schedule Generation</h3>

In [59]:

APs = init_APs(ap_df)
Locations = init_Locations(location_df)




In [60]:
# # JG: NEWMAIN

# date = pd.to_datetime(STARTDATE)
# enddate = pd.to_datetime(ENDDATE)
# dates = pd.date_range(date, enddate)

# fgi = FGI(labor=FGI_STAFFING_BYSHIFT, CPJ=FGI_CPJ)

# for loc_name, loc in Locations.items():
#     fgi.add_Location(loc_name, loc)

# for date in dates:

#     for LN, AP in APs.items():
#         if AP.get_FAROdate().normalize() == pd.to_datetime(date).normalize() and AP.get_LN() not in fgi.APs:
#             fgi.add_ap(AP, date=date)

#     if pd.to_datetime(date).weekday() < 5:
#         for shift in [1,2,3]:
#             for team in ['structure', 'systems', 'declam', 'test']:
#                 max_btg = fgi.get_labor(shift, team) / fgi.CPJ[team]
#                 fgi.assign_labor(team, max_btg_completion=max_btg)

#     fgi.process_move_queue(date=date)
#     fgi.record_day(date)

In [61]:
## MAIN ALGORITHM
# ============================= INITIALIZATION =================================

date = pd.to_datetime(STARTDATE)
enddate = pd.to_datetime(ENDDATE)
dates = pd.date_range(date, enddate)
TERMINATION = False

fgi = FGI(labor=FGI_STAFFING_BYSHIFT, CPJ=FGI_CPJ)

for loc_name, loc in Locations.items():
    fgi.add_Location(loc_name, loc)

# initialize FGI object and add APs and Locations


————————————————————————————————————————————————————————————————————————————————————————————————————

Location D1 added to FGI
Current locations in FGI: ['D1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location D2 added to FGI
Current locations in FGI: ['D1', 'D2']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A1 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A2 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1', 'A2']
——————————————————

In [62]:
FORECAST_UNTIL_COMPLETION = False

In [71]:

# ============================= MAIN SCHEDULING LOOP ============================
while TERMINATION == False:
    if CODECELL_OUTPUT:
        header(f'{date}', '=')
    if date > enddate:
        if CODECELL_OUTPUT: 
            header('END DATE REACHED', '>')
        if not FORECAST_UNTIL_COMPLETION:
            TERMINATION = True
            header('TERMINATION', '==>')
        else:
            header('FORECASTING MODE','>')
        line('>')

    # ————————| ROLLOUT APs FROM FA | —————————————————————————————————————————————————
    for LN, AP in APs.items():
        if LN not in fgi.APs and date == pd.to_datetime(AP.get_FAROdate()):
            if CODECELL_OUTPUT: 
                header('RO')
                print(f'AP {LN} rolled out from FA on {date.strftime("%Y-%m-%d")}')
                print('Added to FGI object')
                line()

            fgi.add_ap(LN, AP)

    # ————————| STATUS UPDATES | —————————————————————————————————————————————————
    # Calendar days only




    


    shift = 0
    while shift < 3:
        # ————————| STATUS UPDATES | —————————————————————————————————————————————————
        if shift == 0:
            for AP in fgi.APs:
                AP.toB1R -= 1

            #fgi.prioritize_queues[QUEUING_RULE]

            if date.weekday() < 5:
                if CODECELL_OUTPUT:
                    print('Workday')
                shift += 1
            else:
                shift = 4
            
            pass # -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   
        
        # ————————| LABOR COMPLETION | —————————————————————————————————————————————————
        else:
            if CODECELL_OUTPUT:
                header(f'Shift: {shift}')
                header('Labor allocation', '-')
            for team in FGI_TEAMS:

                btg_remaining = fgi.get_btg_completion(team, shift)
                labor_queue = fgi.queues['labor'][team]

                if CODECELL_OUTPUT:
                    
                    print(f'Labor for {team}\nBTG: {btg_remaining} | Queue: {labor_queue}')

                while labor_queue is not [] and btg_remaining > 0:
                    for LN in labor_queue:
                        btg_remaining = fgi.APs[LN].complete_BTG(btg_remaining)
                    break

        # fgi.assign_labor(shift)
        shift += 1
    
    # —————————————— MOVE APs (3rd shift, non-buffer)——————————————————————————————————————————————
    if shift == 3 and date not in PLANNED_BUFFER_DAYS:
        header(f'Shift: {shift}')
        header('Move Priority', '-')
        # . . . 

        header('Move Allocation', '-')
        # . . . 
    break

    






=======================================| 2024-06-10 00:00:00 |=======================================
Workday
____________________________________________| Shift: 2 |____________________________________________
----------------------------------------| Labor allocation |----------------------------------------
Labor for structure
BTG: 1.0 | Queue: []
Labor for systems
BTG: 4.0 | Queue: []
Labor for declam
BTG: 0.6666666666666666 | Queue: []
Labor for test
BTG: 0.0 | Queue: []
____________________________________________| Shift: 3 |____________________________________________
------------------------------------------| Move Priority |------------------------------------------
-----------------------------------------| Move Allocation |-----------------------------------------


In [ ]:
fgi.labor[1]['structure']

16